In [1]:
from google.colab import drive
drive.flush_and_unmount()
print('All changes made in this session have been flushed and Drive is unmounted.')

All changes made in this session have been flushed and Drive is unmounted.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ==============================================================================
# Step 0: Mount Google Drive
# ==============================================================================
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")
    exit()

# ==============================================================================
# Step 1: Import Libraries and Configure Paths
# ==============================================================================
import pandas as pd
import sqlite3
import os
from tqdm import tqdm

print("\n--- Colab ETL Script: Load Raw OCR Data ---")

# Define file paths using Colab's directory structure
gdrive_root = "/content/drive/My Drive"
pienza_root = os.path.join(gdrive_root, "_Pienza")
assets_root = os.path.join(pienza_root, 'Assets', 'Database')
staging_area_path = os.path.join(assets_root, '02_Staging_Data')
source_file = os.path.join(staging_area_path, 'staged_raw_ocr.parquet')
db_file = os.path.join(assets_root, 'opus.db')

print(f"Target Database: {db_file}")
print(f"Source Parquet File: {source_file}")

# ==============================================================================
# Step 2: EXTRACT
# ==============================================================================
print("\n--- [E] EXTRACTING staged data from Parquet file ---")
try:
    df_staged = pd.read_parquet(source_file)
    print(f"✅ Success: Extracted {len(df_staged)} rows from source.")
except FileNotFoundError:
    print(f"❌ ERROR: Source file not found at '{source_file}'.")
    print("   -> Please ensure the staging notebook has run successfully and the file exists.")
    exit()
except Exception as e:
    print(f"❌ An unexpected error occurred during extraction: {e}")
    exit()

# ==============================================================================
# Step 3: TRANSFORM (Align Schema)
# ==============================================================================
print("\n--- [T] TRANSFORMING data to match the 'raw_offers_ocr' database schema ---")
try:
    df_to_load = pd.DataFrame()

    # This block maps the columns from the Parquet file to the final database column names
    df_to_load['image_filename'] = df_staged['image_filename']
    df_to_load['time_taken'] = df_staged['time_taken_raw']
    df_to_load['ride_type'] = df_staged['ride_type_raw']
    df_to_load['upfront_fare'] = df_staged['price'] # Use the clean numerical column
    df_to_load['time_to_pickup_min'] = df_staged['time_to_pickup_min']
    df_to_load['dist_to_pickup_km'] = df_staged['dist_to_pickup_km']
    df_to_load['pickup_address'] = df_staged['pickup_loc_raw']
    df_to_load['est_trip_time_min'] = df_staged['est_trip_time_min']
    df_to_load['est_trip_dist_km'] = df_staged['est_trip_dist_km']
    df_to_load['dropoff_address'] = df_staged['dest_address_raw']
    df_to_load['rider_rating'] = df_staged['rider_rating_raw']
    df_to_load['special_note'] = df_staged['special_note_raw']

    print("✅ Success: Schema alignment and column renaming complete.")
except KeyError as e:
    print(f"❌ ERROR: A required column was not found in the source Parquet file: {e}")
    exit()
except Exception as e:
    print(f"❌ An unexpected error occurred during transformation: {e}")
    exit()

# ==============================================================================
# Step 4: LOAD
# ==============================================================================
print(f"\n--- [L] LOADING data into the 'raw_offers_ocr' table ---")
table_name = 'raw_offers_ocr'
conn = sqlite3.connect(db_file)

try:
    # Use 'replace' for development to ensure a clean table on every run
    # This will DROP the table if it exists and create a new one.
    print(f"Writing {len(df_to_load)} rows to table '{table_name}'. This may take a moment...")

    df_to_load.to_sql(
        table_name,
        conn,
        if_exists='replace',
        index=False # Do not write the Pandas index as a column
    )

    print(f"✅ Success: Load operation complete for '{table_name}'.")
except sqlite3.OperationalError as e:
    print(f"❌ An SQL Operational Error occurred. This often means a schema mismatch.")
    print(f"   -> Details: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during the load operation: {e}")
finally:
    if conn:
        conn.close()
        print("Database connection closed.")

# ==============================================================================
# Step 5: VALIDATE
# ==============================================================================
print("\n--- VALIDATING the load operation ---")
conn = sqlite3.connect(db_file)
try:
    count_validate_df = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table_name}", conn)
    total_rows = count_validate_df['count'][0]

    print(f"Total records in '{table_name}' table: {total_rows}")

    if total_rows == len(df_to_load):
        print("✅ Row count matches source DataFrame. Validation successful.")
        print("\n--- Sample of 5 records from the live database ---")
        df_validate = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
        # Using .to_string() for better formatting of wide tables in Colab
        print(df_validate.to_string())
    else:
        print(f"❌ VALIDATION FAILED: Row count mismatch! Expected {len(df_to_load)}, found {total_rows}.")

except Exception as e:
    print(f"❌ An error occurred during validation: {e}")
finally:
    if conn:
        conn.close()
        print("Database connection for validation closed.")

Mounted at /content/drive
✅ Google Drive mounted successfully.

--- Colab ETL Script: Load Raw OCR Data ---
Target Database: /content/drive/My Drive/_Pienza/Assets/Database/opus.db
Source Parquet File: /content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/staged_raw_ocr.parquet

--- [E] EXTRACTING staged data from Parquet file ---
✅ Success: Extracted 4776 rows from source.

--- [T] TRANSFORMING data to match the 'raw_offers_ocr' database schema ---
✅ Success: Schema alignment and column renaming complete.

--- [L] LOADING data into the 'raw_offers_ocr' table ---
Writing 4776 rows to table 'raw_offers_ocr'. This may take a moment...
✅ Success: Load operation complete for 'raw_offers_ocr'.
Database connection closed.

--- VALIDATING the load operation ---
Total records in 'raw_offers_ocr' table: 4776
✅ Row count matches source DataFrame. Validation successful.

--- Sample of 5 records from the live database ---
  image_filename time_taken                ride_type  upfront_fare

In [ ]:
import pandas as pd
import sqlite3
import os

print("--- Colab ETL Script: Load GTS Trip Events ---")

# ==============================================================================
# Step 1: Configuration
# ==============================================================================
# Define paths using Colab's directory structure
gdrive_root = "/content/drive/My Drive"
pienza_root = os.path.join(gdrive_root, "_Pienza")
assets_root = os.path.join(pienza_root, 'Assets', 'Database')
staging_area_path = os.path.join(assets_root, '02_Staging_Data')
db_file = os.path.join(assets_root, 'opus.db')
source_file = os.path.join(staging_area_path, 'staged_gts_events.parquet')

df_gts = None # Initialize variable to be safe

# ==============================================================================
# Step 2: EXTRACT
# ==============================================================================
print("\n--- [E] EXTRACTING data from all sources ---")
try:
    # A. Extract the staged GTS data from the Parquet file
    df_gts = pd.read_parquet(source_file)
    print(f"✅ Success: Extracted {len(df_gts)} event rows from staged Parquet file.")

    # B. Extract key tables from our database for enrichment
    conn = sqlite3.connect(db_file)
    # CRITICAL ASSUMPTION: 'rideID' from GTS matches 'obs_id' in our final polished 'offers' table
    df_offers_keys = pd.read_sql_query("SELECT offer_id, session_id, obs_id FROM offers", conn)
    print(f"✅ Success: Extracted {len(df_offers_keys)} key rows from 'offers' table.")

    df_event_types = pd.read_sql_query("SELECT event_type_id, event_name FROM event_types", conn)
    print(f"✅ Success: Extracted {len(df_event_types)} rows from 'event_types' lookup table.")

except FileNotFoundError:
    print(f"❌ ERROR: Source file not found at '{source_file}'. Please ensure the staging notebook has been run successfully.")
except Exception as e:
    print(f"❌ ERROR during extraction: {e}")
finally:
    if conn:
        conn.close()

# ==============================================================================
# Step 3: TRANSFORM (Only if Extraction was successful)
# ==============================================================================
if df_gts is not None:
    print("\n--- [T] TRANSFORMING and enriching data ---")

    # A. Data Cleaning and Type Conversion
    df_gts['event_timestamp'] = pd.to_datetime(df_gts['realTimestamp'], errors='coerce')
    df_gts['event_timestamp'].fillna(pd.to_datetime(df_gts['serverTimestamp'], errors='coerce'), inplace=True)
    df_gts['latitude'] = pd.to_numeric(df_gts['latitude'], errors='coerce')
    df_gts['longitude'] = pd.to_numeric(df_gts['longitude'], errors='coerce')
    df_gts['upfront_fare'] = pd.to_numeric(df_gts['upfrontFare'], errors='coerce')
    df_gts['realized_fare'] = pd.to_numeric(df_gts['realizedFare'], errors='coerce')
    print("Cleaned data types.")

    # B. Filtering by Date
    cutoff_date = pd.to_datetime('2025-10-02')
    df_gts_filtered = df_gts[df_gts['event_timestamp'] < cutoff_date].copy()
    print(f"Filtered for events before {cutoff_date}. {len(df_gts_filtered)} rows remain.")

    # C. Enriching with Keys via Merging (the JOINs)
    df_enriched = pd.merge(df_gts_filtered, df_offers_keys, left_on='rideID', right_on='obs_id', how='left')
    print("Enriched with offer_id and session_id.")

    df_enriched = pd.merge(df_enriched, df_event_types, left_on='eventType', right_on='event_name', how='left')
    print("Enriched with event_type_id.")

    # D. Final Schema Alignment
    df_to_load = df_enriched[[
        'event_timestamp', 'latitude', 'longitude', 'addressText',
        'upfront_fare', 'realized_fare', 'source',
        'offer_id', 'session_id', 'event_type_id'
    ]].copy()
    # Rename for perfect DB match
    df_to_load.rename(columns={'addressText': 'address_text'}, inplace=True)
    print("Final schema alignment complete.")

    # ==============================================================================
    # Step 4: LOAD
    # ==============================================================================
    print(f"\n--- [L] LOADING data into 'trip_events' table ---")
    conn = sqlite3.connect(db_file)
    try:
        # Use append since this data should be added, not replace old data.
        df_to_load.to_sql('trip_events', conn, if_exists='append', index=False)
        print(f"✅ SUCCESS! Loaded {len(df_to_load)} records into 'trip_events'.")
    except Exception as e:
        print(f"❌ ERROR during load operation: {e}")
    finally:
        if conn:
            conn.close()

else:
    print("\n--- Halting execution due to error in EXTRACT phase. ---")

--- Colab ETL Script: Load GTS Trip Events ---

--- [E] EXTRACTING data from all sources ---
❌ ERROR: Source file not found at '/content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/staged_gts_events.parquet'. Please ensure the staging notebook has been run successfully.


NameError: name 'conn' is not defined